In [22]:
import os
import torch
import cv2
import numpy as np
import random
from torchvision import datasets, transforms
from PIL import Image

class preprocessing_baseline:
    def __init__(self, threshold=128):
        self.threshold = threshold
    def __call__(self, img):
        img_np = np.array(img.convert("L").resize((64, 64)))  
        _, binarized = cv2.threshold(img_np, self.threshold, 255, cv2.THRESH_BINARY)
        return binarized

class AddRandomNoise:
    def __init__(self, candidate_ratios=[0.05, 0.10, 0.25, 0.50]):
        self.candidate_ratios = candidate_ratios

    def __call__(self, img_np):
        h, w = img_np.shape
        noise_ratio = random.choice(self.candidate_ratios)
        num_noisy = int(h * w * noise_ratio)
        if num_noisy == 0:
            return img_np
        coords = np.random.choice(h * w, num_noisy, replace=False)
        y, x = np.unravel_index(coords, (h, w))
        img_np[y, x] = 255 - img_np[y, x]
        return img_np

class MedianFilter:
    def __init__(self, ksize=3):
        self.ksize = ksize

    def __call__(self, img_np):
        return cv2.medianBlur(img_np, self.ksize)

class ComponentFilter:
    def __init__(self, min_area=30):
        self.min_area = min_area

    def __call__(self, img_np):
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(img_np, connectivity=8)
        filtered = np.zeros_like(img_np)
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] >= self.min_area:
                filtered[labels == i] = 255
        return filtered

class MajorityFilter:
    def __init__(self, ksize=3):
        self.ksize = ksize

    def __call__(self, img_np):
        h, w = img_np.shape
        pad = self.ksize // 2
        padded = np.pad(img_np, pad, mode='constant', constant_values=0)
        filtered = np.zeros_like(img_np)

        for y in range(h):
            for x in range(w):
                window = padded[y:y+self.ksize, x:x+self.ksize]
                count_white = np.sum(window == 255)
                count_black = self.ksize * self.ksize - count_white
                filtered[y, x] = 255 if count_white > count_black else 0
        return filtered

class ToTensor:
    def __call__(self, img_np):
        pil_img = Image.fromarray(img_np)
        return transforms.ToTensor()(pil_img)

In [23]:
size_transform = transforms.Compose([
    preprocessing_baseline(threshold=128),
    AddRandomNoise(),
    #MedianFilter(ksize=3),
    #ComponentFilter(),
    MajorityFilter(ksize=3),
    ToTensor()
])

trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train', transform=size_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=40, shuffle=True, num_workers=2, drop_last=True) 

# of trainset = 8000
# of trainloader = 200
input size: torch.Size([1, 64, 64]), 2


In [24]:
import numpy as np 
import torch.optim as optim    
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs, save_path='./prob2_major_weight'):
    os.makedirs(save_path, exist_ok=True)  
    history = np.zeros((0, 3))  # [epoch, avg_loss, avg_accuracy]

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss, correct, total = 0, 0, 0

        for X, y in trainloader: 
            X = X.to(device);y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total
        history = np.vstack((history, [epoch + 1, avg_loss, avg_accuracy]))
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")

        if (epoch + 1) % 10 == 0:
            filename = os.path.join(save_path, f"weight{epoch+1}.pth")
            torch.save(model.state_dict(), filename)
            print(f"Saved checkpoint: {filename}")

    return history
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [25]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
prob2_1 = models.resnet18()
num_ftrs = prob2_1.fc.in_features
prob2_1.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
prob2_1.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)

prob2_1 = prob2_1.to(device)

  2%|▊                                        | 1/50 [01:53<1:32:40, 113.49s/it]

Epoch [1/50] - Loss: 1.352327, Accuracy: 0.3518


  4%|█▋                                       | 2/50 [03:44<1:29:49, 112.28s/it]

Epoch [2/50] - Loss: 1.272269, Accuracy: 0.4019


  6%|██▍                                      | 3/50 [05:36<1:27:48, 112.10s/it]

...

Epoch [48/50] - Loss: 0.410498, Accuracy: 0.7839


 98%|███████████████████████████████████████▏| 49/50 [1:28:54<01:48, 108.16s/it]

Epoch [49/50] - Loss: 0.427618, Accuracy: 0.7825


100%|████████████████████████████████████████| 50/50 [1:30:43<00:00, 108.88s/it]

Epoch [50/50] - Loss: 0.406571, Accuracy: 0.7925
✅ Saved checkpoint: ./prob2_major_weight/weight50.pth


In [ ]:
# 하이퍼파라미터 설정
num_epochs = 50
lr = 0.001
optimizer = torch.optim.Adam(prob2_1.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()
history = train(prob2_1, device, trainloader, optimizer, criterion, num_epochs)

In [37]:
import torch
import os

class AddNoise:
    def __init__(self, noise_ratio):
        self.noise_ratio = noise_ratio

    def __call__(self, img):
        assert isinstance(img, np.ndarray) 
        img_np = img.copy()
        h, w = img_np.shape
        num_pixels = h * w
        num_noisy = int(num_pixels * self.noise_ratio)
        coords = np.random.choice(num_pixels, num_noisy, replace=False)
        y, x = np.unravel_index(coords, (h, w))
        img_np[y, x] = 255 - img_np[y, x]
        return img_np
        
test_transform = transforms.Compose([
    #OURS(size=64, threshold=128, crop=2, method='Otsu'),
    Binarize(threshold=128),
    AddNoise(noise_ratio=0.05), #0.05, 0.10, 0.25, 0.50
    #MedianFilter(ksize=3),
    ComponentFilter(),
    #MajorityFilter(ksize=3),
    ToTensor()
])

testset  = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test',  transform=test_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size = 40, shuffle=True, num_workers=2, drop_last=True)

# 저장된 모델 파일들
model_paths = [
    './prob2_component_weight/weight10.pth',
    './prob2_component_weight/weight20.pth',
    './prob2_component_weight/weight30.pth',
    './prob2_component_weight/weight40.pth',
    './prob2_component_weight/weight50.pth',
]

# 모델 테스트 반복
for path in model_paths:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = models.resnet18()
    num_ftrs = model.fc.in_features
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4)  # Sequential로 동일하게 유지
)
    model.load_state_dict(torch.load(path))
    model = model.to(device)

    test_result = test(model, device, testloader, criterion)

100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 17.28it/s]


test loss : 1.2648 / test_accuracy : 0.4987


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 17.73it/s]


test loss : 1.4562 / test_accuracy : 0.6145


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 17.60it/s]


test loss : 1.9940 / test_accuracy : 0.5974


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 17.32it/s]


test loss : 1.7568 / test_accuracy : 0.6447


100%|███████████████████████████████████████████| 19/19 [00:01<00:00, 17.42it/s]

test loss : 1.9707 / test_accuracy : 0.6250
